In [1]:
import numpy as np
from scipy.special import jv, jn_zeros
from scipy.integrate import quad
from scipy.sparse.linalg import eigsh
from scipy.sparse import diags
from scipy.interpolate import UnivariateSpline
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# =============================================
# 1. Parámetros Físicos
# =============================================
D0 = 2e-9           # Coeficiente de difusión base [m²/s]
R = 1e-6            # Radio del cilindro [m]
z_c = 10e-9         # Altura crítica [m]
lambda_param = 0.2  # Parámetro de decaimiento [nm⁻¹]
kappa = 0.0         # Tasa de adsorción superficial [s⁻¹]
rho0 = 0.75e-6      # Posición radial inicial [m]
z0 = 15e-9          # Posición axial inicial [m]
n_modes_radial = 10 # Modos radiales
m_max = 10          # Modos axiales
N_z = 800          # Nodos en z
alpha_grid = 10.0   # Parámetro de densidad de malla
delta = 1.0e-9      # Ancho de transición

# =============================================
# 2. Generación de Malla Adaptativa
# =============================================
j = np.arange(N_z)
z = z_c * (1 + np.sinh(alpha_grid*(j/N_z - 0.5)) / np.sinh(0.5*alpha_grid))

# =============================================
# 3. Perfil de Difusión D(z)
# =============================================
def D(z):
    S = 0.5*(1 + np.tanh((z - z_c)/delta))
    return D0*(np.exp(-lambda_param*z)*(1 - S) + S)

D_values = D(z)

# =============================================
# 4. Solución Radial (Funciones de Bessel)
# =============================================
alpha_zeros = jn_zeros(0, n_modes_radial)
alpha_n = alpha_zeros/R

# Cálculo de normas radiales
norm_J0_sq = np.zeros(n_modes_radial)
for n in range(n_modes_radial):
    integrand = lambda r: r*jv(0, alpha_n[n]*r)**2
    norm_J0_sq[n] = quad(integrand, 0.01*R, 0.99*R, epsabs=1e-20, epsrel=1e-6)[0]

# =============================================
# 5. Solución Axial (Autovalores/autovectores)
# =============================================
eigenvalues_dict = {}
eigenvectors_dict = {}

for n in range(n_modes_radial):
    main_diag = np.zeros(N_z)
    upper_diag = np.zeros(N_z-1)
    lower_diag = np.zeros(N_z-1)
    
    # Construcción de la matriz
    for j in range(1, N_z-1):
        h_prev = z[j] - z[j-1]
        h_next = z[j+1] - z[j]
        
        D_prev = 2*D_values[j]*D_values[j-1]/(D_values[j] + D_values[j-1] + 1e-30)
        D_next = 2*D_values[j]*D_values[j+1]/(D_values[j] + D_values[j+1] + 1e-30)
        
        coeff_prev = D_prev/(h_prev*(h_prev + h_next)/2)
        coeff_next = D_next/(h_next*(h_prev + h_next)/2)
        
        main_diag[j] = coeff_prev + coeff_next + alpha_n[n]**2*D_values[j]
        upper_diag[j] = -coeff_next
        lower_diag[j-1] = -coeff_prev

    # Condiciones de frontera
    h0 = z[1] - z[0]
    main_diag[0] = D_values[0]/h0 + 1e-10*D0
    upper_diag[0] = -D_values[0]/h0
    
    main_diag[-1] = 1.0  # Dirichlet
    main_diag += 1e-10*D0  # Estabilización
    
    A = diags([main_diag, upper_diag, lower_diag], [0, 1, -1], format='csr')
    
    # Solución numérica
    eigenvalues, eigenvectors = eigsh(A, k=m_max, which='SA', 
                                     tol=1e-10, maxiter=200000, ncv=250)
    eigenvalues = np.clip(eigenvalues, 1e-20, 1e10)  # Límites físicos
    
    eigenvalues_dict[n] = eigenvalues
    eigenvectors_dict[n] = eigenvectors

# =============================================
# 6. Coeficientes A_nm
# =============================================
def compute_A_nm(n):
    A_nm = np.zeros(m_max)
    Z_norms = np.zeros(m_max)
    
    for m in range(m_max):
        integrand = lambda z_val: (UnivariateSpline(z, eigenvectors_dict[n][:, m], s=0)(z_val)**2 / D(z_val))
        Z_norms[m], _ = quad(integrand, z[0], z[-1], epsabs=1e-20, epsrel=1e-8, limit=1000)
    
    Z_m_z0 = np.array([UnivariateSpline(z, eigenvectors_dict[n][:, m], s=0)(z0) for m in range(m_max)])
    J0_rho0 = jv(0, alpha_n[n]*rho0)
    
    valid_modes = (Z_norms > 1e-25) & (np.abs(Z_m_z0) > 1e-12)
    A_nm = (J0_rho0 * Z_m_z0 * valid_modes) / (2*np.pi * rho0 * norm_J0_sq[n] * (Z_norms + 1e-30))
    
    return A_nm

A_nm = Parallel(n_jobs=-1)(delayed(compute_A_nm)(n) for n in range(n_modes_radial))

# =============================================
# 7. Cálculo de G(t)
# =============================================
def compute_G(t, N_quad=10):
    rho_nodes, rho_weights = np.polynomial.legendre.leggauss(N_quad)
    rho_nodes = 0.1*R + 0.8*R*(rho_nodes + 1)/2
    rho_weights *= 0.4*R
    
    z_nodes, z_weights = np.polynomial.legendre.leggauss(N_quad)
    z_nodes = z_c - 5e-9 + 10e-9*(z_nodes + 1)/2
    z_weights *= 5e-9
    
    Z_interps = [[UnivariateSpline(z, eigenvectors_dict[n][:, m], s=1e-12) for m in range(m_max)] for n in range(n_modes_radial)]
    
    total = 0.0
    for i in range(N_quad):
        rho_i = rho_nodes[i]
        for j in range(N_quad):
            z_i = z_nodes[j]
            r2 = rho_i**2 + z_i**2
            f_ij = (3*z_i**2 - rho_i**2)/(r2**2.5 + 1e-30)
            
            sum_term = 0.0
            for n in range(n_modes_radial):
                J0 = jv(0, alpha_n[n]*rho_i)
                decay = np.exp(-(alpha_n[n]**2*D0 + eigenvalues_dict[n])*t)
                Z_m = np.array([Z_interps[n][m](z_i) for m in range(m_max)])
                sum_term += np.sum(A_nm[n] * J0 * Z_m * decay)
            
            total += rho_weights[i] * z_weights[j] * f_ij * sum_term
    
    return (5/(16*np.pi)) * np.clip(total, 1e-40, 1e10)

# =============================================
# 8. Visualizaciones
# =============================================
def plot_all_results():
    # Gráficas de modos radiales
    plt.figure(figsize=(10, 6))
    rho_plot = np.linspace(0.01*R, 0.99*R, 200)
    for n in range(4):
        J0 = jv(0, alpha_n[n]*rho_plot)
        plt.plot(rho_plot/R, J0/np.max(J0), label=f'n={n+1}')
    plt.xlabel(r'$\rho/R$', fontsize=12)
    plt.ylabel(r'$R_n(\rho)$', fontsize=12)
    plt.title('Modos Radiales Normalizados', fontsize=14)
    plt.legend()
    plt.grid(True, ls='--', alpha=0.7)
    plt.savefig('radial_modes_v4.pdf', bbox_inches='tight')
    plt.close()
    
    # Gráficas de modos axiales
    fig, axs = plt.subplots(2, 2, figsize=(12, 8))
    for n in range(2):
        for m in range(2):
            mode = eigenvectors_dict[n][:, m]
            axs[n,m].plot(z*1e9, mode/np.max(np.abs(mode)))
            axs[n,m].set_xlabel('z (nm)', fontsize=10)
            axs[n,m].set_title(f'Modo (n={n}, m={m})')
    plt.tight_layout()
    plt.savefig('axial_modes_v4.pdf', bbox_inches='tight')
    plt.close()
    
    # Decaimiento temporal
    t_plot = np.logspace(-9, -3, 100)
    plt.figure(figsize=(10, 6))
    for n in range(2):
        for m in range(2):
            decay = np.exp(-(alpha_n[n]**2*D0 + eigenvalues_dict[n][m])*t_plot)
            plt.loglog(t_plot, decay, label=f'n={n}, m={m}')
    plt.xlabel('Tiempo (s)', fontsize=12)
    plt.ylabel(r'$T_{nm}(t)$', fontsize=12)
    plt.title('Decaimiento Temporal', fontsize=14)
    plt.legend()
    plt.grid(True, which='both', ls='--')
    plt.savefig('temporal_decay_v4.pdf', bbox_inches='tight')
    plt.close()
    
    # Densidad de probabilidad 3D
    def plot_3d_density(t):
        rho_vals = np.linspace(0.1*R, 0.9*R, 40)
        z_vals = np.linspace(z_c-5e-9, z_c+5e-9, 40)
        Rg, Zg = np.meshgrid(rho_vals, z_vals)
        P = np.zeros_like(Rg)
        
        for i in range(40):
            for j in range(40):
                sum_p = 0.0
                for n in range(n_modes_radial):
                    J0 = jv(0, alpha_n[n]*Rg[i,j])
                    decay = np.exp(-(alpha_n[n]**2*D0 + eigenvalues_dict[n][0])*t)
                    Z = UnivariateSpline(z, eigenvectors_dict[n][:, 0], s=0)(Zg[i,j])
                    sum_p += A_nm[n][0] * J0 * Z * decay
                P[i,j] = np.abs(sum_p)
        
        fig = plt.figure(figsize=(14, 10))
        ax = fig.add_subplot(111, projection='3d')
        surf = ax.plot_surface(Rg*1e6, Zg*1e9, P, cmap='viridis', 
                             rstride=1, cstride=1, alpha=0.8)
        ax.set_xlabel(r'$\rho$ (μm)', fontsize=12)
        ax.set_ylabel('z (nm)', fontsize=12)
        ax.set_zlabel(r'$P(\rho,z,t)$', fontsize=12)
        plt.title(f'Densidad de Probabilidad en t = {t:.1e} s', fontsize=14)
        fig.colorbar(surf, shrink=0.6, aspect=20)
        plt.savefig(f'density_t_{t:.0e}_v4.pdf', bbox_inches='tight')
        plt.close()
    
    # Generar para tiempos clave
    for t in [1e-9, 1e-6, 1e-3]:
        plot_3d_density(t)

# =============================================
# 9. Ejecución Principal
# =============================================
if __name__ == "__main__":
    plot_all_results()
    
    # Cálculo de G(t)
    t_values = np.logspace(-10, -1, 50)
    G_values = Parallel(n_jobs=-1,prefer='threads')(delayed(compute_G)(t) for t in t_values)
    
    # Guardar datos y gráfico final
    np.savetxt('G_vs_t_v4.dat', np.column_stack((t_values, G_values)),
               header='Tiempo(s) G(t)', fmt='%.3e')
    
    plt.figure(figsize=(10, 6))
    plt.loglog(t_values, G_values, 'bo-', label='Simulación')
    plt.plot(t_values, 1e-8*t_values**-1.5, 'r--', label='Teoría $t^{-3/2}$')
    plt.xlabel('Tiempo (s)', fontsize=12)
    plt.ylabel('G(t)', fontsize=12)
    plt.title('Función de Correlación Nano-NMR', fontsize=14)
    plt.legend()
    plt.grid(True, which='both', ls='--')
    plt.savefig('Gt_result_v4.pdf', bbox_inches='tight')
    plt.close()

ArpackNoConvergence: ARPACK error -1: No convergence (200001 iterations, 5/10 eigenvectors converged)